# Regroupements et tables pivots

Ce TP va avec deux chapitres : « Regroupements » (parties 1 à 4) et « Tables pivots et tableaux croisés » (parties 5 et 6).

- Partie 1 — Données d'exemple `ventes` : les formes de sortie de `agg`, `transform`, `filter` et `apply`, vérifiables à la main
- Partie 2 — Agrégations : le prix au m² des logements, par département
- Partie 3 — Index hiérarchiques : régions et départements
- Partie 4 — `transform`, `filter` et `apply` : écart à la médiane, ventes aberrantes, ventes les plus chères
- Partie 5 — Tables pivots : le prix au m² par région et par année
- Partie 6 — Tableaux croisés et `melt` : les défauts de paiement de clients d'une banque

Les parties 2 à 5 s'enchaînent sur les mêmes données : chacune reprend les variables des précédentes, que la solution de chaque exercice définit. Les exercices marqués « facultatif » ne servent pas à la suite : passez-les si le temps manque.

## Données et bibliothèques

- [DVF](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) (Demandes de valeurs foncières, Etalab, Licence Ouverte 2.0) : les ventes immobilières en France de janvier 2014 à juin 2022, une ligne par mutation, dans la version nettoyée de [NyxAether/DVF](https://github.com/NyxAether/DVF), en Parquet partitionné par année.
- [Départements de France](https://www.data.gouv.fr/datasets/departements-de-france) (data.gouv.fr, ODbL) : chaque département et sa région.
- [Default of Credit Card Clients](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients) (UCI, CC BY 4.0) : 30 000 clients d'une banque taïwanaise en 2005.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-france.csv
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/credit-default/credit-default.csv

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Partie 1 — Données d'exemple `ventes`

Le jeu des slides, une ligne par vente : assez petit pour vérifier chaque résultat à la main. `forme` affiche ce qu'il faut regarder d'un résultat : son type, sa forme (lignes, colonnes), les noms des niveaux de son index et ses trois premières étiquettes.

In [ ]:
ventes = pd.DataFrame(
  [
    ("Nord", "Lille", "vélo", "2026-01", 3, 1500),
    ("Nord", "Lille", "casque", "2026-01", 10, 400),
    ("Nord", "Amiens", "vélo", "2026-02", 2, 900),
    ("Sud", "Nice", "vélo", "2026-01", 5, 2600),
    ("Sud", "Nice", "casque", "2026-02", 8, 300),
    ("Sud", "Marseille", "casque", "2026-02", 6, 250),
    ("Ouest", "Nantes", "vélo", "2026-01", 4, 2000),
    ("Ouest", "Rennes", "casque", "2026-02", 7, 290),
    ("Ouest", "Nantes", "casque", "2026-02", 3, 120),
    ("Nord", "Amiens", "casque", "2026-01", 5, 200),
  ],
  columns=["region", "ville", "produit", "mois", "quantite", "montant"],
)


def forme(resultat):
  """Type, forme, niveaux et début de l'index de resultat."""
  index = resultat.index
  print(type(resultat).__name__, resultat.shape, list(index.names), list(index[:3]))


ventes

### 1.1 Prédire les formes de sortie

Avec `g = ventes.groupby("region")`, six calculs :

1. `g["montant"].sum()`
2. `g.agg(n=("montant", "size"), ca=("montant", "sum"))`
3. `g["montant"].transform("sum")`
4. `g.head(1)`
5. `g.filter(lambda d: len(d) > 3)`
6. `g.apply(lambda d: d.nlargest(2, "montant"), include_groups=False)`

Pour chacun, prédisez d'abord, sans l'exécuter : `Series` ou `DataFrame`, le nombre de lignes et de colonnes, et l'index (les régions, l'index d'origine de `ventes`, ou les deux). Notez vos réponses en commentaire, puis passez chaque résultat à `forme`.

Réussi quand vos six prédictions sont justes ; sinon, relisez la frame « Ce que reçoit et renvoie la fonction » avant de passer à la suite.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les règles des slides :

- une ligne par groupe, indexée par la clé : les agrégations (`sum`, `agg`) ;
- l'index d'origine, toutes les lignes : les transformations (`transform`, `cumsum`, `rank`) ;
- l'index d'origine, une partie des lignes : les filtres (`head`, `filter`) ;
- `apply` : selon ce que renvoie la fonction ; un `DataFrame` par groupe donne leur concaténation, avec la clé en niveau d'index ajouté devant l'index d'origine ;
- `include_groups=False` : la colonne clé n'est pas passée à la fonction, donc absente du résultat.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
g = ventes.groupby("region")
forme(g["montant"].sum())
forme(g.agg(n=("montant", "size"), ca=("montant", "sum")))
forme(...)
```

#### Solution

In [ ]:
g = ventes.groupby("region")
forme(g["montant"].sum())  # Series (3,) : une valeur par région
forme(g.agg(n=("montant", "size"), ca=("montant", "sum")))  # DataFrame (3, 2)
forme(g["montant"].transform("sum"))  # Series (10,) : index d'origine
forme(g.head(1))  # DataFrame (3, 6) : lignes 0, 3 et 6 de ventes
forme(g.filter(lambda d: len(d) > 3))  # DataFrame (4, 6) : le Nord seul
# DataFrame (6, 5) : MultiIndex (region, index d'origine), sans la colonne region
forme(g.apply(lambda d: d.nlargest(2, "montant"), include_groups=False))

### 1.2 Agrégations nommées par produit

Pour chaque produit, calculez en un seul `agg`, avec des agrégations nommées, le nombre de ventes `n`, la quantité vendue `quantite` et le chiffre d'affaires `ca`. Ajoutez ensuite la colonne `prix_unitaire` : le chiffre d'affaires divisé par la quantité.

Pour vérifier : 6 ventes de casques, 39 casques pour 1 560 €, soit 40 € le casque ; 500 € le vélo.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `ventes.groupby("produit").agg(nom=(colonne, fonction), ...)` : une colonne par agrégation nommée, dans l'ordre écrit ;
- `size` compte les lignes du groupe, `sum` additionne ses valeurs ;
- `assign(prix_unitaire=...)` : le rapport de deux sommes se calcule après l'agrégation. La moyenne des prix unitaires de chaque vente donnerait autre chose (492,50 € pour un vélo), chaque vente comptant autant qu'une autre, quelle que soit sa quantité.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
par_produit = ventes.groupby("produit").agg(
  n=("montant", ...),
  quantite=(..., "sum"),
  ca=...,
)
par_produit.assign(prix_unitaire=...)
```

#### Solution

In [ ]:
par_produit = ventes.groupby("produit").agg(
  n=("montant", "size"),
  quantite=("quantite", "sum"),
  ca=("montant", "sum"),
)
par_produit.assign(prix_unitaire=par_produit["ca"] / par_produit["quantite"])

### 1.3 `transform` et `filter`

Ajoutez à une copie de `ventes` (`assign`) la colonne `part_ville` : la part de chaque vente dans le chiffre d'affaires de sa ville, avec `transform`.

Gardez ensuite les ventes des régions dont la vente moyenne dépasse 800 €, de deux façons : avec `filter`, puis avec un masque booléen calculé par `transform`. Vérifiez que les deux résultats sont égaux avec `equals`.

Pour vérifier : `part_ville` vaut 1 pour les ventes de Marseille et de Rennes, seules dans leur ville, et ses valeurs somment à 1 dans chaque ville ; il reste 6 ventes, celles de l'Ouest et du Sud.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `ventes.groupby("ville")["montant"].transform("sum")` : le chiffre d'affaires de la ville, répété sur chacune de ses ventes, à l'index de `ventes` ;
- `g.filter(lambda d: ...)` : la fonction reçoit le `DataFrame` d'un groupe et renvoie un booléen ;
- `g["montant"].transform("mean") > 800` : le même critère, en masque aligné sur `ventes` ;
- `a.equals(b)` : mêmes valeurs, même index, mêmes types.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
ca_ville = ventes.groupby("ville")["montant"].transform(...)
avec_part = ventes.assign(part_ville=...)
print(avec_part.groupby("ville")["part_ville"].sum())

g = ventes.groupby("region")
gardees = g.filter(lambda d: ...)
masque = g["montant"].transform(...) > 800
print(len(gardees), ventes[masque].equals(gardees))
```

#### Solution

In [ ]:
ca_ville = ventes.groupby("ville")["montant"].transform("sum")
avec_part = ventes.assign(part_ville=ventes["montant"] / ca_ville)
print(avec_part.groupby("ville")["part_ville"].sum())
print(avec_part.round(2))

g = ventes.groupby("region")
gardees = g.filter(lambda d: d["montant"].mean() > 800)
masque = g["montant"].transform("mean") > 800
print(len(gardees), ventes[masque].equals(gardees))

## Partie 2 — Agrégations

DVF a une ligne par mutation (vente, échange, adjudication…) : sa date, sa nature, sa valeur foncière, le nombre de maisons, d'appartements et de dépendances vendus, leurs surfaces, leur adresse… Chargé en entier, il occupe 4,4 Go en mémoire : on n'en lit que ce qui sert.

### 2.1 Charger seulement ce qui sert

Chargez dans `dvf` les colonnes `COLONNES` des seules ventes (`nature_mutation` égal à `"Vente"`) de `dvf/dvf-linearized-2014-2022.parquet`, avec `pd.read_parquet` et ses paramètres `columns` et `filters` : le filtre s'applique pendant la lecture, même sur une colonne qu'on ne charge pas.

Affichez la mémoire occupée par chaque colonne, avec `memory_usage(deep=True)` : que pèse l'index, les identifiants des mutations en chaînes de caractères ? Remplacez-le par un index d'entiers avec `reset_index`, puis affichez le bilan de `info(memory_usage="deep")`.

Pour vérifier : 9 859 162 ventes ; environ 735 Mo, dont 566 Mo d'index, puis 169 Mo.

In [ ]:
COLONNES = [
  "annee_mutation",
  "code_departement",
  "valeur_fonciere",
  "nombre_maisons",
  "nombre_appartements",
  "surface_reelle_bati_total",
]
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `filters=[("nature_mutation", "==", "Vente")]` : une liste de conditions `(colonne, opérateur, valeur)`, qui doivent toutes être vraies ;
- `dvf.memory_usage(deep=True)` : les octets de chaque colonne, l'index en tête ; `/ 2**20` les convertit en Mo ;
- `dvf.reset_index(drop=True)` : `drop=True` jette l'ancien index au lieu d'en faire une colonne.

Un squelette à copier dans la cellule de l'exercice, sous `COLONNES`, en remplaçant chaque `...` :

```python
dvf = pd.read_parquet(
  "dvf/dvf-linearized-2014-2022.parquet",
  columns=...,
  filters=[(..., "==", ...)],
)
print(dvf.memory_usage(deep=True) / 2**20)
dvf = dvf.reset_index(...)
dvf.info(memory_usage=...)
```

#### Solution

In [ ]:
dvf = pd.read_parquet(
  "dvf/dvf-linearized-2014-2022.parquet",
  columns=COLONNES,
  filters=[("nature_mutation", "==", "Vente")],
)
print(dvf.memory_usage(deep=True) / 2**20)
dvf = dvf.reset_index(drop=True)
dvf.info(memory_usage="deep")

### 2.2 Des codes de département sur deux caractères

Les codes de département de DVF n'ont pas de zéro initial (`"1"` pour l'Ain), alors que la table des départements de la partie 3 en a un (`"01"`). `code_departement` est catégorielle : 97 valeurs distinctes, et un petit entier par ligne. Renommez ses catégories, avec `cat.rename_categories`, plutôt que ses 9,9 millions de valeurs, puis triez-les avec `cat.reorder_categories` : l'ordre des catégories est celui des résultats de `groupby`.

Pour vérifier : les catégories commencent par `01`, `02`, `03` et finissent par `971` à `974` ; la colonne pèse toujours 9,4 Mo environ.

**Documentation utile**

- [`Series.cat.rename_categories`](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.rename_categories.html)
- [`Series.cat.reorder_categories`](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.reorder_categories.html)
- [`str.zfill`](https://docs.python.org/3/library/stdtypes.html#str.zfill)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `rename_categories` prend un dictionnaire ou une fonction appliquée à chaque catégorie : `lambda c: c.zfill(2)` complète à gauche par des zéros jusqu'à deux caractères (`"2A"` et `"971"` ne changent pas) ;
- après le renommage, les catégories restent dans l'ordre d'avant (`01`, `10`, `11`…, `02`…) ; `reorder_categories(sorted(...))` reçoit toutes les catégories, dans le nouvel ordre ;
- `replace` sur une colonne catégorielle est obsolète (`FutureWarning`), et `astype(str).str.zfill(2)` créerait 9,9 millions de chaînes, soit environ 480 Mo.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
codes = dvf["code_departement"].cat.rename_categories(lambda c: ...)
dvf["code_departement"] = codes.cat.reorder_categories(sorted(...))
print(dvf["code_departement"].cat.categories)
print(dvf["code_departement"].memory_usage(deep=True) / 2**20)
```

#### Solution

In [ ]:
codes = dvf["code_departement"].cat.rename_categories(lambda c: c.zfill(2))
dvf["code_departement"] = codes.cat.reorder_categories(sorted(codes.cat.categories))
print(dvf["code_departement"].cat.categories)
print(dvf["code_departement"].memory_usage(deep=True) / 2**20)

### 2.3 Le prix au m² des ventes d'un seul logement

Une mutation peut vendre un immeuble entier : la moyenne des valeurs foncières d'un département ne dit rien du prix d'un logement. Gardez dans `biens` les ventes d'un seul logement (une maison ou un appartement, avec ou sans dépendances), avec les colonnes `annee_mutation` et `code_departement`, et deux nouvelles :

- `type`, catégorielle : `"maison"` ou `"appartement"` ;
- `prix_m2` : la valeur foncière divisée par la surface bâtie, manquante (`NaN`) quand elle sort de l'intervalle de 100 à 50 000 €/m² (surface nulle, prix symbolique, erreur de saisie).

Supprimez ensuite `dvf` (`del dvf`) pour libérer sa mémoire.

Pour vérifier : 6 074 486 ventes, dont 3 226 956 maisons ; 6 044 907 prix au m², de médiane 2 246 €/m².

**Documentation utile**

- [`Series.between`](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html)
- [`numpy.where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `dvf["nombre_maisons"] + dvf["nombre_appartements"] == 1` : le masque des ventes d'un seul logement ;
- `np.where(condition, "maison", "appartement")`, dans `pd.Categorical(...)` pour une colonne catégorielle ;
- `prix.where(prix.between(100, 50_000))` : `NaN` hors de l'intervalle (bornes comprises) ; une surface nulle donne un prix infini, hors de l'intervalle lui aussi ;
- `dvf.assign(type=..., prix_m2=...).loc[masque, colonnes]` : les nouvelles colonnes, puis les lignes et les colonnes gardées, sans modifier `dvf`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
un_logement = ...
prix = dvf["valeur_fonciere"] / dvf[...]
biens = dvf.assign(
  type=pd.Categorical(np.where(..., "maison", "appartement")),
  prix_m2=prix.where(...),
).loc[un_logement, ["annee_mutation", "code_departement", "type", "prix_m2"]]
del dvf
print(len(biens), biens["type"].value_counts(), biens["prix_m2"].describe())
```

#### Solution

In [ ]:
un_logement = dvf["nombre_maisons"] + dvf["nombre_appartements"] == 1
prix = dvf["valeur_fonciere"] / dvf["surface_reelle_bati_total"]
biens = dvf.assign(
  type=pd.Categorical(np.where(dvf["nombre_maisons"] == 1, "maison", "appartement")),
  prix_m2=prix.where(prix.between(100, 50_000)),
).loc[un_logement, ["annee_mutation", "code_departement", "type", "prix_m2"]]
del dvf, prix, un_logement
print(len(biens))
print(biens["type"].value_counts())
print(biens["prix_m2"].describe())

### 2.4 `size` contre `count`

Par année, comptez dans un même `agg` les ventes de `biens` avec `size` et les prix au m² utilisables avec `count`, puis ajoutez leur différence, la colonne `manquants`. `annee_mutation` est catégorielle : passez `observed=True` au `groupby`.

Pour vérifier : 29 579 prix manquants en tout, environ 0,5 % des ventes de chaque année ; 2022 n'a que son premier semestre.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `biens.groupby("annee_mutation", observed=True)["prix_m2"].agg(["size", "count"])` : une colonne par fonction ;
- `size` compte les lignes, `NaN` compris ; `count` les valeurs non manquantes ;
- `.assign(manquants=lambda t: t["size"] - t["count"])` ; `.sum()` donne les totaux.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
comptes = biens.groupby(..., observed=True)["prix_m2"].agg([...]).assign(manquants=...)
print(comptes)
print(comptes.sum())
```

#### Solution

In [ ]:
comptes = (
  biens.groupby("annee_mutation", observed=True)["prix_m2"]
  .agg(["size", "count"])
  .assign(manquants=lambda t: t["size"] - t["count"])
)
print(comptes)
print(comptes.sum())

### 2.5 Agrégations nommées par département

Par département, calculez en un seul `agg`, avec des agrégations nommées :

- `n`, le nombre de prix au m² utilisables ;
- `mediane`, leur médiane ;
- `iqr`, leur écart interquartile (troisième quartile moins premier), avec une fonction `lambda`.

Triez par médiane décroissante, puis tracez en barres horizontales la médiane des 15 départements les plus chers. Pourquoi la médiane plutôt que la moyenne ?

Pour vérifier : 97 départements ; Paris en tête (médiane de 9 360 €/m², écart interquartile de 3 496 €/m²), devant les Hauts-de-Seine (6 158 €/m²) ; la Creuse en dernier (684 €/m²).

**Documentation utile**

- [`DataFrame.plot.barh`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.barh.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `n=("prix_m2", "count")` : `count`, pas `size`, qui compterait aussi les prix manquants ;
- `iqr=("prix_m2", lambda s: s.quantile(0.75) - s.quantile(0.25))` : la fonction reçoit la `Series` d'un groupe ;
- `observed=True` : `code_departement` est catégorielle ;
- `sort_values("mediane", ascending=False)`, puis `head(15)["mediane"].plot.barh()` ; `ax.invert_yaxis()` met le premier en haut ; `plt.show()` affiche la figure ;
- la médiane résiste aux prix extrêmes qui restent (une vente à 50 000 €/m² pèse autant qu'une autre, pas 20 fois plus).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
stats = (
  biens.groupby("code_departement", observed=True)
  .agg(
    n=("prix_m2", ...),
    mediane=...,
    iqr=("prix_m2", lambda s: ...),
  )
  .sort_values(...)
)
print(stats.round())
ax = stats.head(15)["mediane"].plot.barh()
ax.invert_yaxis()
plt.show()
```

#### Solution

In [ ]:
stats = (
  biens.groupby("code_departement", observed=True)
  .agg(
    n=("prix_m2", "count"),
    mediane=("prix_m2", "median"),
    iqr=("prix_m2", lambda s: s.quantile(0.75) - s.quantile(0.25)),
  )
  .sort_values("mediane", ascending=False)
)
print(stats.round())
ax = stats.head(15)["mediane"].plot.barh(figsize=(6, 5))
ax.invert_yaxis()
ax.set_xlabel("Prix médian (€/m²)")
ax.set_ylabel("Département")
plt.show()

## Partie 3 — Index hiérarchiques

`regions` relie chaque code de département à sa région : une `Series` indexée par code, lue de la table des départements (en chaînes, pour garder les zéros initiaux).

In [ ]:
regions = pd.read_csv("departements-france.csv", dtype=str).set_index(
  "code_departement"
)["nom_region"]
regions.head()

### 3.1 La région de chaque vente

Ajoutez à `biens` la colonne catégorielle `region`, en traduisant `code_departement` par `map` avec `regions`.

Pour vérifier : aucune région manquante ; 17 régions (Mayotte n'est pas dans DVF, ni l'Alsace et la Moselle, qui ont leur propre livre foncier) ; 1 172 978 ventes en Île-de-France.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `biens["code_departement"].map(regions)` : sur une colonne catégorielle, `map` ne traduit que les 97 catégories ; plusieurs départements ayant la même région, le résultat est en chaînes : `.astype("category")` ;
- `isna().sum()`, `nunique()` et `value_counts()` pour vérifier ;
- c'est une jointure sur une seule colonne : `merge`, du chapitre « Jointures », ferait de même avec une table entière.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
biens["region"] = biens["code_departement"].map(...).astype(...)
print(biens["region"].isna().sum(), biens["region"].nunique())
biens["region"].value_counts()
```

#### Solution

In [ ]:
biens["region"] = biens["code_departement"].map(regions).astype("category")
print(biens["region"].isna().sum(), biens["region"].nunique())
biens["region"].value_counts()

### 3.2 Région, puis département

Regroupez `biens` par région puis par département pour obtenir `par_dep`, indexé par un `MultiIndex` à deux niveaux, avec les colonnes `n` et `mediane` de 2.5. Puis :

1. les départements de Bretagne, avec `loc` ;
2. la ligne de Paris (`"75"`), sans donner sa région, avec `xs` ;
3. le département le plus cher de chaque région : un tri, puis `groupby(level=...)` et `tail` ;
4. le nombre de prix au m² utilisables de chaque région, la somme de `n` avec `groupby(level=...)`.

Pour vérifier : 97 lignes (essayez aussi `observed=False` : combien de lignes, et pourquoi ?) ; Bretagne : 22, 29, 35 et 56 ; les plus chers : 75 en Île-de-France, 06 en Provence-Alpes-Côte d'Azur, 74 en Auvergne-Rhône-Alpes ; 4 589 prix au m² en Guyane.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `biens.groupby(["region", "code_departement"], observed=True).agg(...)` : avec `observed=False`, toutes les combinaisons des catégories, 17 × 97 = 1 649 lignes, presque toutes vides ;
- `par_dep.loc["Bretagne"]` : le premier niveau ; `par_dep.xs("75", level="code_departement")` : un autre niveau ;
- les niveaux de `par_dep` sont catégoriels : `groupby(level="region", observed=True)`, sinon `FutureWarning` ;
- `par_dep.sort_values("mediane").groupby(level="region", observed=True).tail(1)` : la dernière ligne, la plus chère, de chaque région.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
par_dep = biens.groupby([..., ...], observed=True).agg(
  n=("prix_m2", "count"), mediane=("prix_m2", "median")
)
print(len(par_dep))
print(par_dep.loc[...])
print(par_dep.xs(..., level=...))
print(par_dep.sort_values(...).groupby(level=..., observed=True).tail(...))
print(par_dep.groupby(level=..., observed=True)["n"].sum())
```

#### Solution

In [ ]:
par_dep = biens.groupby(["region", "code_departement"], observed=True).agg(
  n=("prix_m2", "count"), mediane=("prix_m2", "median")
)
print(len(par_dep))
print(par_dep.loc["Bretagne"])
print(par_dep.xs("75", level="code_departement"))
print(par_dep.sort_values("mediane").groupby(level="region", observed=True).tail(1))
print(par_dep.groupby(level="region", observed=True)["n"].sum())

### 3.3 Maisons et appartements par région (facultatif)

Comptez les ventes par région et par type, passez le type en colonnes avec `unstack`, triez les régions par part d'appartements, et tracez des barres horizontales empilées.

Pour vérifier : 71 % d'appartements en Provence-Alpes-Côte d'Azur et en Île-de-France, 69 % en Corse, 21 % dans les Hauts-de-France.

**Documentation utile**

- [`DataFrame.plot.barh`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.barh.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `biens.groupby(["region", "type"], observed=True).size().unstack("type")` : une colonne par type ;
- `par_type["appartement"] / par_type.sum(axis=1)` : la part d'appartements de chaque région ;
- `par_type.loc[part.sort_values().index]` : les lignes dans l'ordre des parts ;
- `plot.barh(stacked=True)` : une barre par ligne, une couleur par colonne.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
par_type = biens.groupby([...], observed=True).size().unstack(...)
part = par_type["appartement"] / ...
print(part.sort_values().round(2))
par_type.loc[part.sort_values().index].plot.barh(stacked=...)
plt.show()
```

#### Solution

In [ ]:
par_type = biens.groupby(["region", "type"], observed=True).size().unstack("type")
part = par_type["appartement"] / par_type.sum(axis=1)
print(part.sort_values().round(2))
ax = par_type.loc[part.sort_values().index].plot.barh(stacked=True, figsize=(7, 6))
ax.set_xlabel("Ventes d'un seul logement, 2014-2022")
plt.show()

## Partie 4 — `transform`, `filter` et `apply`

### 4.1 Écart à la médiane du département, ventes aberrantes

Ajoutez à `biens`, avec `transform` :

- `ecart`, l'écart relatif du prix au m² à la médiane de son département (0,25 : 25 % plus cher que cette médiane) ;
- `aberrant`, vrai quand le prix au m² est plus de trois fois au-dessus ou en dessous de cette médiane.

Appliquez ensuite la même règle avec la médiane nationale : quelle part des ventes de Paris (75) et de la Creuse (23) chaque règle déclare-t-elle aberrante ?

Pour vérifier : 4,3 % de ventes aberrantes selon la règle par département ; Paris, 86 % selon la règle nationale contre 2,7 % ; la Creuse, 54 % contre 8,8 %.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `biens.groupby("code_departement", observed=True)["prix_m2"].transform("median")` : la médiane du département, sur chaque ligne de `biens` ;
- `ratio = biens["prix_m2"] / mediane`, puis `ratio - 1` et `(ratio < 1 / 3) | (ratio > 3)` ; une comparaison avec `NaN` est fausse ;
- `biens["prix_m2"].median()` : la médiane nationale, un nombre, que la division diffuse sur toutes les lignes ;
- la moyenne d'une colonne booléenne est la part de `True` : `parts.groupby(biens["code_departement"], observed=True).mean()`, avec `parts` un `DataFrame` des deux règles.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
mediane = biens.groupby(...)["prix_m2"].transform(...)
ratio = biens["prix_m2"] / mediane
biens["ecart"] = ...
biens["aberrant"] = ...
ratio_national = biens["prix_m2"] / ...
parts = pd.DataFrame(
  {
    "national": ...,
    "departement": biens["aberrant"],
  }
)
print(biens["aberrant"].mean())
parts.groupby(biens["code_departement"], observed=True).mean().loc[["75", "23"]]
```

#### Solution

In [ ]:
mediane = biens.groupby("code_departement", observed=True)["prix_m2"].transform(
  "median"
)
ratio = biens["prix_m2"] / mediane
biens["ecart"] = ratio - 1
biens["aberrant"] = (ratio < 1 / 3) | (ratio > 3)
ratio_national = biens["prix_m2"] / biens["prix_m2"].median()
parts = pd.DataFrame(
  {
    "national": (ratio_national < 1 / 3) | (ratio_national > 3),
    "departement": biens["aberrant"],
  }
)
print(biens["aberrant"].mean())
parts.groupby(biens["code_departement"], observed=True).mean().loc[["75", "23"]]

### 4.2 `filter` contre un masque

On n'étudie le prix des appartements que dans les départements qui en comptent au moins 2 000 ventes. À partir d'`appartements`, gardez les ventes de ces départements de deux façons : avec `filter`, puis avec un masque calculé par `transform("size")`. Vérifiez que les deux résultats sont égaux, chronométrez-les avec `%time`, et listez les départements exclus.

Pour vérifier : 7 départements exclus, 9 540 ventes en moins ; la Creuse en compte le moins (571).

In [ ]:
appartements = biens[biens["type"] == "appartement"]
g = appartements.groupby("code_departement", observed=True)
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `%time x = expression` : le temps d'une instruction ;
- `g.filter(lambda d: len(d) >= 2000)` : un appel de fonction Python par groupe ;
- `g["prix_m2"].transform("size") >= 2000` : le nombre de ventes du département sur chaque ligne, en un seul calcul ; le masque se combine avec d'autres conditions (`&`) ;
- `n_ventes = g.size()`, puis `n_ventes[n_ventes < 2000]`.

Un squelette à copier dans la cellule de l'exercice, sous `g`, en remplaçant chaque `...` :

```python
%time gardes = g.filter(lambda d: ...)
%time masque = g["prix_m2"].transform(...) >= 2000
print(appartements[masque].equals(gardes), len(appartements) - len(gardes))
n_ventes = g.size()
n_ventes[...]
```

#### Solution

In [ ]:
%time gardes = g.filter(lambda d: len(d) >= 2000)
%time masque = g["prix_m2"].transform("size") >= 2000
print(appartements[masque].equals(gardes), len(appartements) - len(gardes))
n_ventes = g.size()
n_ventes[n_ventes < 2000].sort_values()

### 4.3 Les ventes les plus chères au m² (facultatif)

Trouvez les trois ventes au prix au m² le plus élevé de chaque département, de deux façons :

- `top3_apply` : `apply` d'une fonction qui renvoie les trois plus grandes lignes de son groupe (`nlargest`), avec `include_groups=False` ;
- `top3_tri` : un tri de tout `biens` par prix décroissant, puis `groupby(...).head(3)`.

Chronométrez-les avec `%time`, par département, puis par département, année et type. Trouvent-elles les mêmes prix ? Laquelle passe le mieux à l'échelle quand le nombre de groupes augmente, et pourquoi ?

Pour vérifier : 291 lignes par département (97 groupes × 3), 5 236 par département, année et type (1 746 groupes, dont quelques-uns de moins de 3 ventes), avec les mêmes prix (des ex æquo peuvent changer les lignes choisies) ; `apply` est le plus rapide avec 97 groupes, le tri avec 1 746 : le temps d'`apply` croît avec le nombre de groupes, celui du tri dépend surtout du nombre de lignes.

In [ ]:
def top3_apply(cles): ...


def top3_tri(cles): ...


# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `biens.groupby(cles, observed=True).apply(lambda d: d.nlargest(3, "prix_m2"), include_groups=False)` : un appel Python par groupe, 97, puis 1 746 ;
- `biens.sort_values("prix_m2", ascending=False).groupby(cles, observed=True).head(3)` : un seul tri des 6 millions de lignes, quels que soient les groupes ;
- `np.array_equal(np.sort(a["prix_m2"]), np.sort(b["prix_m2"]))` : mêmes prix, dans n'importe quel ordre ;
- la méthode intégrée `groupby(cles)["prix_m2"].nlargest(3)` est la plus rapide, mais ne renvoie que la colonne des prix.

Un squelette à copier dans la cellule de l'exercice, à la place des deux fonctions, en remplaçant chaque `...` :

```python
def top3_apply(cles):
  g = biens.groupby(cles, observed=True)
  return g.apply(lambda d: ..., include_groups=False)


def top3_tri(cles):
  tri = biens.sort_values(...)
  return tri.groupby(cles, observed=True).head(...)


%time a = top3_apply("code_departement")
%time b = top3_tri("code_departement")
print(len(a), len(b), np.array_equal(np.sort(a["prix_m2"]), np.sort(b["prix_m2"])))
%time a = top3_apply(["code_departement", "annee_mutation", "type"])
%time b = top3_tri(...)
print(...)
```

#### Solution

In [ ]:
def top3_apply(cles):
  g = biens.groupby(cles, observed=True)
  return g.apply(lambda d: d.nlargest(3, "prix_m2"), include_groups=False)


def top3_tri(cles):
  tri = biens.sort_values("prix_m2", ascending=False)
  return tri.groupby(cles, observed=True).head(3)


%time a = top3_apply("code_departement")
%time b = top3_tri("code_departement")
print(len(a), len(b), np.array_equal(np.sort(a["prix_m2"]), np.sort(b["prix_m2"])))
%time a = top3_apply(["code_departement", "annee_mutation", "type"])
%time b = top3_tri(["code_departement", "annee_mutation", "type"])
print(len(a), len(b), np.array_equal(np.sort(a["prix_m2"]), np.sort(b["prix_m2"])))

## Partie 5 — Tables pivots

### 5.1 Le prix médian par région et par année

Avec `pivot_table`, calculez `prix_region` : le prix au m² médian de `biens` par région (en lignes) et par année (en colonnes), avec les totaux (`margins`), nommés `"France"`. `region` et `annee_mutation` sont catégorielles : `observed=True`.

Pour vérifier : 18 lignes et 10 colonnes ; la case France × France est la médiane nationale de 2.3, 2 246 €/m² : un total se calcule sur les ventes, pas sur les cases ; l'Île-de-France passe de 4 036 €/m² en 2014 à 5 022 en 2022.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `index="region"`, `columns="annee_mutation"`, `values="prix_m2"`, `aggfunc="median"` ;
- `margins=True, margins_name="France"` : une ligne et une colonne de totaux ;
- `.round()` pour l'affichage.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
prix_region = biens.pivot_table(
  index=...,
  columns=...,
  values=...,
  aggfunc=...,
  margins=...,
  margins_name=...,
  observed=True,
)
prix_region.round()
```

#### Solution

In [ ]:
prix_region = biens.pivot_table(
  index="region",
  columns="annee_mutation",
  values="prix_m2",
  aggfunc="median",
  margins=True,
  margins_name="France",
  observed=True,
)
prix_region.round()

### 5.2 L'évolution d'une année sur l'autre

Calculez `evolution`, l'évolution relative du prix médian de chaque région d'une année sur l'autre, avec `pct_change(axis=1)`, sur `prix_region` sans ses totaux. Affichez-la en heatmap, annotée en pourcentages et centrée sur 0. Quelle région a le plus augmenté de 2014 à 2021 ?

Pour vérifier : la colonne 2014 est vide (pas d'année précédente) ; en 2021, hausse partout, de 2 % (Corse) à 11 % (Pays de la Loire) ; de 2014 à 2021, Pays de la Loire (+28,6 %) et Bretagne (+28,2 %) en tête. Prudence avec 2022 : son premier semestre seulement.

**Documentation utile**

- [`DataFrame.pct_change`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pct_change.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `prix_region.drop(index="France", columns="France")` : sans les totaux ;
- `pct_change(axis=1)` : (année − année précédente) / année précédente, le long des colonnes ;
- `sns.heatmap(evolution, annot=True, fmt=".0%", cmap="RdBu_r", center=0)` : `fmt` formate les annotations, `center=0` met le blanc à 0 ;
- `prix_region[2021] / prix_region[2014] - 1` : l'évolution de 2014 à 2021.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
evolution = prix_region.drop(...).pct_change(...)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(evolution, annot=True, fmt=..., cmap="RdBu_r", center=..., ax=ax)
plt.show()
(prix_region[2021] / prix_region[2014] - 1).sort_values()
```

#### Solution

In [ ]:
evolution = prix_region.drop(index="France", columns="France").pct_change(axis=1)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(evolution, annot=True, fmt=".0%", cmap="RdBu_r", center=0, ax=ax)
plt.show()
(prix_region[2021] / prix_region[2014] - 1).sort_values()

### 5.3 `pivot` refuse les doublons (facultatif)

`medianes` est le prix médian par région, année et type de logement, en format long. Essayez d'en faire un tableau région × année avec `pivot` : pourquoi l'erreur ? Comptez les couples (région, année) en double avec `duplicated`. Puis :

1. faites-le avec `pivot_table` et `aggfunc="mean"`, comme le ferait un tableur, et comparez avec `prix_region` : pourquoi ce résultat est-il faux ?
2. remodelez `medianes` sans rien perdre, le type dans l'index de `pivot`.

Pour vérifier : `ValueError: Index contains duplicate entries` ; 153 doublons ; la moyenne des médianes des maisons et des appartements dépasse la vraie médiane de 83 € en Bretagne en 2014 ; 34 lignes (17 régions × 2 types).

**Documentation utile**

- [`DataFrame.duplicated`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)

In [ ]:
medianes = (
  biens.groupby(["region", "annee_mutation", "type"], observed=True)["prix_m2"]
  .median()
  .reset_index()
)
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `try:` … `except ValueError as erreur:` pour afficher l'erreur sans arrêter l'exécution ;
- `medianes.duplicated(["region", "annee_mutation"]).sum()` : les lignes dont le couple a déjà été vu ;
- `pivot_table(..., aggfunc="mean", observed=True)`, puis la différence avec `prix_region.drop(index="France", columns="France")` ;
- la médiane de toutes les ventes n'est pas la moyenne des médianes de deux groupes de tailles différentes ;
- `pivot(index=["region", "type"], columns="annee_mutation", values="prix_m2")`.

Un squelette à copier dans la cellule de l'exercice, sous `medianes`, en remplaçant chaque `...` :

```python
try:
  medianes.pivot(index=..., columns=..., values=...)
except ValueError as erreur:
  print(erreur)
print(medianes.duplicated(...).sum())
moyennes = medianes.pivot_table(..., observed=True)
print((moyennes - prix_region.drop(index="France", columns="France")).round())
medianes.pivot(index=[...], columns=..., values=...)
```

#### Solution

In [ ]:
try:
  medianes.pivot(index="region", columns="annee_mutation", values="prix_m2")
except ValueError as erreur:
  print(erreur)
print(medianes.duplicated(["region", "annee_mutation"]).sum())
moyennes = medianes.pivot_table(
  index="region",
  columns="annee_mutation",
  values="prix_m2",
  aggfunc="mean",
  observed=True,
)
print((moyennes - prix_region.drop(index="France", columns="France")).round())
medianes.pivot(
  index=["region", "type"], columns="annee_mutation", values="prix_m2"
).round()

## Partie 6 — Tableaux croisés et `melt`

`credit` : 30 000 clients d'une banque taïwanaise, d'avril à septembre 2005, un par ligne. Les colonnes utiles ici :

- `limit` : le plafond de crédit, en dollars taïwanais ;
- `sex`, `education` : codés par des entiers, traduits ci-dessous (les codes 0, 5 et 6 d'`education` ne sont pas documentés) ;
- `age`, en années ;
- `delay_apr` à `delay_sep` : le statut de paiement de chaque mois, d'avril à septembre : -1 payé à temps, 1 à 8 mois de retard, 9 pour neuf mois et plus ; -2 et 0 ne sont pas documentés ;
- `default` : 1 si le client a fait défaut le mois suivant, 0 sinon.

In [ ]:
credit = pd.read_csv("credit-default.csv")
credit["sex"] = credit["sex"].map({1: "homme", 2: "femme"})
credit["education"] = (
  credit["education"]
  .map({1: "master ou doctorat", 2: "université", 3: "lycée", 4: "autre"})
  .fillna("inconnu")
)
credit.head()

### 6.1 Le taux de défaut par niveau d'études

Avec `pd.crosstab`, calculez la part de clients en défaut (`default`) par niveau d'études (`education`) : des lignes qui somment à 1, et la ligne des totaux. Affichez aussi les effectifs : quels taux reposent sur peu de clients ?

Pour vérifier : 22,1 % de défauts en tout ; 25,2 % pour le lycée, 19,2 % pour master ou doctorat ; « autre » et « inconnu » (5,7 % et 7,5 %) ne comptent que 123 et 345 clients.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `pd.crosstab(credit["education"], credit["default"], ...)` : des `Series`, pas des noms de colonnes ;
- `normalize="index"` : chaque ligne somme à 1 ; `margins=True` : la ligne (et la colonne) `All` ;
- sans `normalize`, les effectifs.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
taux = pd.crosstab(credit["education"], ..., normalize=..., margins=...)
print(taux.round(3))
pd.crosstab(..., margins=True)
```

#### Solution

In [ ]:
taux = pd.crosstab(
  credit["education"], credit["default"], normalize="index", margins=True
)
print(taux.round(3))
pd.crosstab(credit["education"], credit["default"], margins=True)

### 6.2 Le taux de défaut par âge et par sexe

Découpez `age` en tranches de dix ans, de `(20, 30]` à `(50, 60]`, puis `(60, 80]`, avec `pd.cut`. Croisez-les avec le sexe : le taux de défaut par tranche et par sexe, avec les totaux, par `crosstab` et ses paramètres `values` et `aggfunc`.

Pour vérifier : les âges vont de 21 à 79 ans ; le taux le plus bas chez les 30-40 ans (20,4 %), et 18,5 % chez les femmes de cet âge ; les hommes font plus défaut que les femmes dans toutes les tranches, sauf les 60-80 ans.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `pd.cut(credit["age"], bins=[20, 30, 40, 50, 60, 80])` : des intervalles fermés à droite, dans une `Series` catégorielle ordonnée ;
- `pd.crosstab(tranche_age, credit["sex"], values=credit["default"], aggfunc="mean", margins=True)` : la moyenne d'une colonne de 0 et de 1 est la part de 1 ;
- `credit["age"].agg(["min", "max"])` pour les bornes.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(credit["age"].agg(["min", "max"]))
tranche_age = pd.cut(credit["age"], bins=...)
pd.crosstab(tranche_age, ..., values=..., aggfunc=..., margins=True).round(3)
```

#### Solution

In [ ]:
print(credit["age"].agg(["min", "max"]))
tranche_age = pd.cut(credit["age"], bins=[20, 30, 40, 50, 60, 80])
pd.crosstab(
  tranche_age, credit["sex"], values=credit["default"], aggfunc="mean", margins=True
).round(3)

### 6.3 Les retards de paiement, du format large au format long

Les six colonnes `delay_apr` à `delay_sep` sont un format large, un mois par colonne. Passez-les en format long avec `melt` dans `long` : les colonnes `id` et `default` gardées, et deux nouvelles, `mois` et `retard`. Retirez le préfixe `delay_` des mois et rendez-les catégoriels, dans l'ordre du calendrier, `MOIS`. Regroupez les retards de 4 mois et plus, puis calculez avec `pivot_table` le taux de défaut par retard (en lignes) et par mois (en colonnes).

Pour vérifier : 180 000 lignes (30 000 × 6) ; à partir de 2 mois de retard, plus d'un client sur deux fait défaut, contre 13 à 20 % pour les statuts -2 à 0 ; un retard d'un mois n'apparaît presque qu'en septembre, une particularité de ces données.

**Documentation utile**

- [`Series.str.removeprefix`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.removeprefix.html)
- [`Series.clip`](https://pandas.pydata.org/docs/reference/api/pandas.Series.clip.html)

In [ ]:
MOIS = ["apr", "may", "jun", "jul", "aug", "sep"]
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `credit.melt(id_vars=["id", "default"], value_vars=[f"delay_{m}" for m in MOIS], var_name="mois", value_name="retard")` ;
- `long["mois"].str.removeprefix("delay_")`, dans `pd.Categorical(..., categories=MOIS, ordered=True)` ;
- `long["retard"].clip(upper=4)` : 4 et plus deviennent 4 ;
- `pivot_table(index="retard", columns="mois", values="default", aggfunc="mean", observed=True)` : `mois` est catégorielle ; une case sans client est `NaN`.

Un squelette à copier dans la cellule de l'exercice, sous `MOIS`, en remplaçant chaque `...` :

```python
long = credit.melt(
  id_vars=[...],
  value_vars=[f"delay_{m}" for m in MOIS],
  var_name=...,
  value_name=...,
)
long["mois"] = pd.Categorical(..., categories=MOIS, ordered=True)
long["retard"] = long["retard"].clip(...)
print(long.shape)
long.pivot_table(index=..., columns=..., values=..., aggfunc=..., observed=True)
```

#### Solution

In [ ]:
long = credit.melt(
  id_vars=["id", "default"],
  value_vars=[f"delay_{m}" for m in MOIS],
  var_name="mois",
  value_name="retard",
)
long["mois"] = pd.Categorical(
  long["mois"].str.removeprefix("delay_"), categories=MOIS, ordered=True
)
long["retard"] = long["retard"].clip(upper=4)
print(long.shape)
long.pivot_table(
  index="retard", columns="mois", values="default", aggfunc="mean", observed=True
).round(2)

### 6.4 `crosstab` ou `pivot_table` (facultatif)

Calculez le plafond de crédit moyen (`limit`) par niveau d'études et par sexe, une fois avec `crosstab` et ses paramètres `values` et `aggfunc`, une fois avec `pivot_table`. Les deux tables sont-elles égales (`equals`) ? Quand préférer l'une ou l'autre ?

Pour vérifier : `True` ; master ou doctorat : 208 557 dollars pour les femmes, 219 251 pour les hommes.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `pd.crosstab(credit["education"], credit["sex"], values=credit["limit"], aggfunc="mean")` ;
- `credit.pivot_table(index="education", columns="sex", values="limit", aggfunc="mean")` ;
- `crosstab` prend des `Series`, qui peuvent venir de tables différentes ou être calculées (`pd.cut`) ; `pivot_table` prend des noms de colonnes d'un seul `DataFrame`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
croise = pd.crosstab(..., ..., values=..., aggfunc=...)
pivot = credit.pivot_table(index=..., columns=..., values=..., aggfunc=...)
print(croise.equals(pivot))
croise.round()
```

#### Solution

In [ ]:
croise = pd.crosstab(
  credit["education"], credit["sex"], values=credit["limit"], aggfunc="mean"
)
pivot = credit.pivot_table(
  index="education", columns="sex", values="limit", aggfunc="mean"
)
print(croise.equals(pivot))
croise.round()